In [ ]:
import heapq
import itertools
from dataclasses import dataclass, field
from typing import Any, Optional

@dataclass(order=True)
class Event:
    sort_index: float = field(init=False, repr=False)
    timestamp: float
    priority: int  # Sequence ID to break ties (FIFO)
    type: str      
    payload: Any  

    def __post_init__(self):
        self.sort_index = self.timestamp

class MarketEngine:
    def __init__(self):
        self.current_time = 0.0
        self.event_queue = [] # The Min-Heap
        self._sequence_counter = itertools.count() # Unique ID generator
        self.is_running = False

    def schedule(self, delay: float, type: str, payload: Any):
       
        arrival_time = self.current_time + delay
        seq_id = next(self._sequence_counter)
        
        event = Event(timestamp=arrival_time, priority=seq_id, type=type, payload=payload)
        
        heapq.heappush(self.event_queue, (arrival_time, seq_id, event))
        
        print(f"[Log] Scheduled {type} at {arrival_time} (Current: {self.current_time})")

    def run(self):
        self.is_running = True
        print("\n--- Simulation Started ---")

        while self.event_queue and self.is_running:
            # 1. Pop the earliest event - O(log N)
            # The tuple is (time, seq_id, EventObject)
            event_time, _, event = heapq.heappop(self.event_queue)

            # 2. Advance the Virtual Clock (Time Jumps!)
            if event_time < self.current_time:
                raise ValueError("Time Travel Error: Event in past!")
            
            self.current_time = event_time

            # 3. Process the Event
            self.handle_event(event)

        print("--- Simulation Ended ---")

    def handle_event(self, event: Event):
        print(f"[{self.current_time:.2f}] Processing: {event.type} | Payload: {event.payload}")

        if event.type == 'MARKET_CLOSE':
            self.is_running = False
            print(f"[{self.current_time:.2f}] Market Closed. Stopping loop.")
            
        elif event.type == 'NEW_ORDER':
            pass 


def run_simulation():
    engine = MarketEngine()

    # Scenario:
    # t=0:  Trader A sends a Buy Order (High Latency: 10ms)
    # t=2:  Trader B sends a Sell Order (Low Latency: 2ms)
    # t=5:  Trader A realizes mistake, sends Cancel (Latency: 2ms)
    # t=20: Market Closes
    
    # Note how we schedule relative to current time (which is 0 right now)
    
    # Trader A: Sends Buy at t=0, Latency=10 -> Arrives t=10
    engine.schedule(delay=10, type='NEW_ORDER', payload={'id': 1, 'side': 'BUY', 'qty': 100})
    
    # Trader B: Sends Sell at t=2, Latency=2 -> Arrives t=4
    # To simulate "Trader B acts at t=2", we cheat slightly here by pre-loading 
    # the queue, but in a real sim, an Agent class would call schedule().
    # Here we manually set the arrival times for demonstration.
     
    engine.schedule(delay=4, type='NEW_ORDER', payload={'id': 2, 'side': 'SELL', 'qty': 100})
    
    # Trader A Cancel: Sent at t=5, Latency=2 -> Arrives t=7
    engine.schedule(delay=7, type='CANCEL_ORDER', payload={'id': 1})

    # Market Close
    engine.schedule(delay=20, type='MARKET_CLOSE', payload=None)

    # Let's inject a tie-breaker test
    # Two orders arriving at exactly t=15
    engine.schedule(delay=15, type='NEW_ORDER', payload={'id': 3, 'desc': 'Tie Breaker 1'})
    engine.schedule(delay=15, type='NEW_ORDER', payload={'id': 4, 'desc': 'Tie Breaker 2'})

    # Start the engine
    engine.run()

if __name__ == "__main__":
    run_simulation()

[Log] Scheduled NEW_ORDER at 10.0 (Current: 0.0)
[Log] Scheduled NEW_ORDER at 4.0 (Current: 0.0)
[Log] Scheduled CANCEL_ORDER at 7.0 (Current: 0.0)
[Log] Scheduled MARKET_CLOSE at 20.0 (Current: 0.0)
[Log] Scheduled NEW_ORDER at 15.0 (Current: 0.0)
[Log] Scheduled NEW_ORDER at 15.0 (Current: 0.0)

--- Simulation Started ---
[4.00] Processing: NEW_ORDER | Payload: {'id': 2, 'side': 'SELL', 'qty': 100}
[7.00] Processing: CANCEL_ORDER | Payload: {'id': 1}
[10.00] Processing: NEW_ORDER | Payload: {'id': 1, 'side': 'BUY', 'qty': 100}
[15.00] Processing: NEW_ORDER | Payload: {'id': 3, 'desc': 'Tie Breaker 1'}
[15.00] Processing: NEW_ORDER | Payload: {'id': 4, 'desc': 'Tie Breaker 2'}
[20.00] Processing: MARKET_CLOSE | Payload: None
[20.00] Market Closed. Stopping loop.
--- Simulation Ended ---
